# E5: Hybrid CNN-RNN for DNA Thermodynamics

**Thesis:** Inductive Biases in Representation Learning for DNA Thermodynamic Property Prediction  
**Experiment ID:** E5  
**Thesis Chapter:** Chapter 4 Spatial vs Sequential Inductive Bias  

## Core Idea

DNA melting is fundamentally a **dual-physics** process:

1. **Sequential unzipping** — paired bases at the ends of the stem fray first, then the process propagates inward like unzipping a zipper. This is captured by a **1D Bidirectional GRU** reading the sequence left-to-right and right-to-left simultaneously.

2. **Spatial base-pairing** — the stability of each position depends on which bases are directly across the H-bond ladder (Watson-Crick pairing) and which bases stack above/below. This is captured by the **2D Folded Ladder CNN** (same as E2).

Neither branch alone captures both effects. We concatenate their latent representations before the MLP head:

$$\mathbf{z}_{\text{hybrid}} = \text{MLP}\!\left([\mathbf{z}_{\text{BiGRU}} \,\|\, \mathbf{z}_{\text{2DCNN}}]\right)$$

### Inductive Biases Being Compared

| Model | Inductive Bias |
|-------|----------------|
| GNN (E0) | Local message passing — permutation equivariant |
| 2D CNN (E2) | Spatial convolution — folded structural view |
| SAT (E3) | Global relational attention + structural adjacency |
| PINN (E4) | Thermodynamic law in loss — physics-constrained |
| **Hybrid (E5, this notebook)** | **Dual modality — sequential unzipping + spatial pairing** |

### Research Question
> Is DNA thermodynamic prediction a dual-modality problem? Does combining sequential (BiGRU) and spatial (2D CNN) representations outperform either branch in isolation?

In [1]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os, json, math, time
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import wandb

import sys
if sys.platform == 'win32' and not os.environ.get('WANDB_MODE'):
    os.environ['WANDB_MODE'] = 'online'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

COLORS = {
    'GNN':    '#7f8c8d',
    '1D_CNN': '#3498db',
    '2D_CNN': '#e74c3c',
    'SAT':    '#9b59b6',
    'PINN':   '#e67e22',
    'Hybrid': '#1abc9c',   # teal for Hybrid CNN-RNN
}
MODEL_NAME = 'Hybrid'

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
MAX_LEN   = 24    # for BiGRU branch  (7 channels × L positions)
MAX_WIDTH = 15    # for 2D CNN branch (6 channels × 3 rows × W)
BASES     = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
NT_MAP    = {'A': 0, 'T': 1, 'G': 2, 'C': 3}   # 2D encoding uses different order

DATA_CSV   = 'data/models/raw/combined_dataset.csv'
SPLIT_JSON = 'data/models/raw/combined_data_split.json'

config = dict(
    model_name      = 'HybridCNNRNN',
    experiment_id   = 'E5',
    # BiGRU branch
    seq_input_dim   = 7,       # 4 one-hot base + 3 one-hot structure
    gru_hidden      = 64,      # 128 total (bidirectional)
    gru_layers      = 2,
    # 2D CNN branch
    cnn_in_channels = 6,
    # Fusion
    fusion_dim      = 256,     # concat of BiGRU(128) + CNN(128)
    dropout         = 0.2,
    # Training
    n_epoch         = 200,
    batch_size      = 256,
    lr              = 1e-3,
    weight_decay    = 1e-5,
    grad_clip       = 1.0,
    dataset         = 'arr',
    norm_method     = 'normalize',
    wandb_project   = 'NNN_Thesis_Experiments',
    checkpoint_dir  = 'MyExperiments/Hybrid/models',
)
print('Config loaded:', config)

Config loaded: {'model_name': 'HybridCNNRNN', 'experiment_id': 'E5', 'seq_input_dim': 7, 'gru_hidden': 64, 'gru_layers': 2, 'cnn_in_channels': 6, 'fusion_dim': 256, 'dropout': 0.2, 'n_epoch': 200, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 1e-05, 'grad_clip': 1.0, 'dataset': 'arr', 'norm_method': 'normalize', 'wandb_project': 'NNN_Thesis_Experiments', 'checkpoint_dir': 'MyExperiments/Hybrid/models'}


In [13]:
# ── 3. Data Loading & Normalization ───────────────────────────────────────────
df = pd.read_csv(DATA_CSV, index_col='SEQID')
df.sort_index(inplace=True)

with open(SPLIT_JSON) as f:
    split = json.load(f)

# Train & evaluate on 'arr' only — lit_uv / ov are held-out generalization sets
TRAIN_DATASET = 'arr'

train_df = df.loc[split['train_ind']].dropna(subset=['dH', 'Tm'])
train_df = train_df[train_df['dataset'] == TRAIN_DATASET]
val_df   = df.loc[split['val_ind']  ].dropna(subset=['dH', 'Tm'])
val_df   = val_df[val_df['dataset'] == TRAIN_DATASET]
test_df  = df.loc[split['test_ind'] ].dropna(subset=['dH', 'Tm'])
test_df  = test_df[test_df['dataset'] == TRAIN_DATASET]

sumstats = {
    'dH_min': float(train_df['dH'].min()), 'dH_max': float(train_df['dH'].max()),
    'Tm_min': float(train_df['Tm'].min()), 'Tm_max': float(train_df['Tm'].max()),
}

def normalize(val, mn, mx):   return (val - mn) / (mx - mn)
def unnormalize(val, mn, mx): return val * (mx - mn) + mn

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')
print(f'dH range: [{sumstats["dH_min"]:.1f}, {sumstats["dH_max"]:.1f}] kcal/mol')
print(f'Tm range: [{sumstats["Tm_min"]:.1f}, {sumstats["Tm_max"]:.1f}] °C')

Train: 25,025  |  Val: 1,318  |  Test: 1,387
dH range: [-68.2, -2.7] kcal/mol
Tm range: [13.6, 68.6] °C


In [4]:
# ── 4. Dual Encoding: 1D sequence + 2D folded ladder ──────────────────────────

# ---- 1D encoding (for BiGRU branch) ----
def encode_1d(seq, struct, max_len=MAX_LEN):
    """
    Returns (max_len, 7): 4 one-hot base + 3 one-hot structure.
    Identical to 1D_CNN_for_dna.ipynb encoding.
    """
    seq_map  = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
    str_map  = {'(': 0, ')': 1, '.': 2}
    struct_c = struct.replace('+', '')
    L        = min(len(seq), max_len)
    x        = np.zeros((max_len, 7), dtype=np.float32)
    for i in range(L):
        if seq[i].upper() in seq_map:
            x[i, seq_map[seq[i].upper()]] = 1.0
        if i < len(struct_c) and struct_c[i] in str_map:
            x[i, 4 + str_map[struct_c[i]]] = 1.0
    return x   # (max_len, 7) — time-first for GRU


# ---- 2D encoding (for CNN branch) — from 2Dconv.ipynb / E2 ----
def encode_2d_hairpin(seq, struct, max_width=MAX_WIDTH):
    n_stem   = struct.count('(')
    n_loop   = struct.count('.')
    half_loop = n_loop // 2
    has_mid  = (n_loop % 2 == 1)
    fold_len = n_stem + half_loop
    top_seq  = seq[:fold_len]
    mid_nt   = seq[fold_len] if has_mid else None
    bot_seq  = seq[fold_len + (1 if has_mid else 0):][::-1]
    hbond    = [1.0] * n_stem + [0.0] * half_loop
    tensor   = np.zeros((6, 3, max_width), dtype=np.float32)
    for i, nt in enumerate(top_seq):
        if nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()], 0, i] = 1.0
    for i, nt in enumerate(bot_seq):
        if nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()], 2, i] = 1.0
    for i, h in enumerate(hbond):
        tensor[4, 1, i] = h
    bb = fold_len
    if bb < max_width:
        tensor[5, :, bb] = 1.0
    if has_mid and mid_nt and mid_nt.upper() in NT_MAP and bb < max_width:
        tensor[NT_MAP[mid_nt.upper()], 1, bb] = 1.0
    return tensor


def encode_2d_duplex(s1, s2, max_width=MAX_WIDTH):
    tensor = np.zeros((6, 3, max_width), dtype=np.float32)
    for i, nt in enumerate(s1):
        if i < max_width and nt.upper() in NT_MAP:
            tensor[NT_MAP[nt.upper()], 0, i] = 1.0
    for i, nt in enumerate(s2[::-1]):
        if i < max_width and nt.upper() in NT_MAP:
            tensor[NT_MAP[nt.upper()], 2, i] = 1.0
    for i in range(min(len(s1), max_width)):
        tensor[4, 1, i] = 1.0
    return tensor


def encode_row_dual(row, max_len=MAX_LEN, max_width=MAX_WIDTH):
    """Returns (x_1d, x_2d) pair for a dataframe row."""
    seq    = str(row['RefSeq'])
    struct = str(row['TargetStruct'])

    x_1d = encode_1d(seq, struct, max_len)   # (L, 7)

    if '+' in struct:
        plus_pos = struct.index('+')
        s1, s2   = seq[:plus_pos], seq[plus_pos+1:]
        x_2d = encode_2d_duplex(s1, s2, max_width)
    else:
        x_2d = encode_2d_hairpin(seq, struct, max_width)

    return x_1d, x_2d


_r   = df.iloc[0]
_1d, _2d = encode_row_dual(_r)
print(f'1D shape: {_1d.shape}  (L × 7 — time-first for BiGRU)')
print(f'2D shape: {_2d.shape}  (6 channels × 3 rows × {MAX_WIDTH} width)')

1D shape: (24, 7)  (L × 7 — time-first for BiGRU)
2D shape: (6, 3, 15)  (6 channels × 3 rows × 15 width)


In [5]:
# ── 5. Dataset & DataLoaders ──────────────────────────────────────────────────

class DNAHybridDataset(Dataset):
    """Returns (x_1d, x_2d, y) for dual-branch model."""

    def __init__(self, df, sumstats):
        self.df       = df.reset_index(drop=False)
        self.sumstats = sumstats

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        x_1d, x_2d = encode_row_dual(row)

        dH_norm = normalize(row['dH'], self.sumstats['dH_min'], self.sumstats['dH_max'])
        Tm_norm = normalize(row['Tm'], self.sumstats['Tm_min'], self.sumstats['Tm_max'])
        y = np.array([dH_norm, Tm_norm], dtype=np.float32)

        return torch.tensor(x_1d), torch.tensor(x_2d), torch.tensor(y)


train_ds = DNAHybridDataset(train_df, sumstats)
val_ds   = DNAHybridDataset(val_df,   sumstats)
test_ds  = DNAHybridDataset(test_df,  sumstats)

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512,                  shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512,                  shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')
_x1, _x2, _y = next(iter(train_loader))
print(f'Batch — x_1d: {_x1.shape}  x_2d: {_x2.shape}  y: {_y.shape}')

Train batches: 98  |  Val batches: 3
Batch — x_1d: torch.Size([256, 24, 7])  x_2d: torch.Size([256, 6, 3, 15])  y: torch.Size([256, 2])


In [6]:
# ── 6. Model: Hybrid CNN-RNN ──────────────────────────────────────────────────
#
# Two completely independent branches, fused at the latent level:
#
#   1D BiGRU branch:
#       (B, L, 7) → BiGRU(hidden=64, layers=2) → mean pool → (B, 128)
#       Captures sequential context: which bases precede/follow a pair
#       (nearest-neighbour stacking parameters, positional effects)
#
#   2D CNN branch:
#       (B, 6, 3, W) → Conv2d blocks → AttentionPool2d → (B, 128)
#       Captures spatial structure: which bases are hydrogen-bonded
#       (pairing partners, loop size, backbone turn)
#
#   Fusion MLP:
#       cat[(B,128),(B,128)] = (B,256) → Linear(256,128) → ReLU → Linear(128,2)

class BiGRUBranch(nn.Module):
    """
    Bidirectional GRU branch for sequential DNA context.
    Input:  (B, L, 7)    — sequence + structure one-hot, time-first
    Output: (B, 2*hidden) — mean-pooled hidden states
    """
    def __init__(self, input_dim=7, hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size  = input_dim,
            hidden_size = hidden,
            num_layers  = n_layers,
            batch_first = True,
            bidirectional = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        """
        x: (B, L, 7)
        Returns: (B, 2*hidden) — mean of all hidden states
        """
        # Mask padding (all-zero rows) so they don't pollute the mean
        not_pad = (x.sum(-1) != 0).float().unsqueeze(-1)   # (B, L, 1)
        out, _ = self.gru(x)                               # (B, L, 2*hidden)
        out    = out * not_pad                             # zero out padding positions
        lengths = not_pad.sum(1).clamp(min=1)              # (B, 1) — keep as (B,1) for broadcast
        return self.drop(out.sum(1) / lengths)             # (B, 2*hidden) / (B, 1) → (B, 2*hidden)


class AttentionPool2d(nn.Module):
    """Learned attention pooling: (B,C,H,W) → (B,C). Identical to E2/E4."""
    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=1), nn.Tanh(),
            nn.Conv2d(64, 1, kernel_size=1),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        w = torch.softmax(self.attn(x).view(B, 1, -1), dim=-1)
        return (x.view(B, C, -1) * w).sum(-1)


class CNN2DBranch(nn.Module):
    """
    2D CNN branch for spatial structure. Backbone identical to E2.
    Input:  (B, 6, 3, W)
    Output: (B, 128)
    """
    def __init__(self, in_channels=6, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64,  (3,3), padding=(1,1)),
            nn.BatchNorm2d(64),  nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(64,        128, (3,3), padding=(1,1)),
            nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(128,       128, (3,5), padding=(1,2)),
            nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout),
        )
        self.pool = AttentionPool2d(128)

    def forward(self, x):
        return self.pool(self.conv(x))   # (B, 128)


class HybridCNNRNN(nn.Module):
    """
    Hybrid model combining 1D BiGRU (sequential) + 2D CNN (spatial) branches.

    The two branches are independent; only the MLP fusion head is shared.
    This isolates the contribution of each modality to the final prediction.
    """

    def __init__(self, seq_input_dim=7, gru_hidden=64, gru_layers=2,
                 cnn_in_channels=6, dropout=0.2):
        super().__init__()
        self.rnn_branch = BiGRUBranch(seq_input_dim, gru_hidden, gru_layers, dropout)
        self.cnn_branch = CNN2DBranch(cnn_in_channels, dropout)

        rnn_out = gru_hidden * 2    # bidirectional → 128
        cnn_out = 128
        fusion  = rnn_out + cnn_out  # 256

        self.fusion_head = nn.Sequential(
            nn.Linear(fusion, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, x_1d, x_2d):
        """
        x_1d: (B, L, 7)       — sequence + structure, time-first
        x_2d: (B, 6, 3, W)    — folded ladder image
        Returns: (B, 2)        — [dH_norm, Tm_norm]
        """
        z_rnn = self.rnn_branch(x_1d)   # (B, 128)
        z_cnn = self.cnn_branch(x_2d)   # (B, 128)
        z     = torch.cat([z_rnn, z_cnn], dim=-1)  # (B, 256)
        return self.fusion_head(z)       # (B, 2)


model = HybridCNNRNN(
    seq_input_dim   = config['seq_input_dim'],
    gru_hidden      = config['gru_hidden'],
    gru_layers      = config['gru_layers'],
    cnn_in_channels = config['cnn_in_channels'],
    dropout         = config['dropout'],
).to(device)

n_params     = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_rnn_params = sum(p.numel() for p in model.rnn_branch.parameters())
n_cnn_params = sum(p.numel() for p in model.cnn_branch.parameters())
print(f'Model: HybridCNNRNN')
print(f'  BiGRU branch:  {n_rnn_params:,} parameters')
print(f'  2D CNN branch: {n_cnn_params:,} parameters')
print(f'  Total:         {n_params:,} parameters')

Model: HybridCNNRNN
  BiGRU branch:  102,528 parameters
  2D CNN branch: 332,225 parameters
  Total:         476,035 parameters


In [7]:
# ── 7. Metrics & Evaluation Helpers (standard across all notebooks) ───────────

def compute_metrics(pred_norm, true_norm, sumstats):
    if torch.is_tensor(pred_norm): pred_norm = pred_norm.cpu().numpy()
    if torch.is_tensor(true_norm): true_norm = true_norm.cpu().numpy()

    dH_p = pred_norm[:, 0] * (sumstats['dH_max'] - sumstats['dH_min']) + sumstats['dH_min']
    Tm_p = pred_norm[:, 1] * (sumstats['Tm_max'] - sumstats['Tm_min']) + sumstats['Tm_min']
    dH_t = true_norm[:, 0] * (sumstats['dH_max'] - sumstats['dH_min']) + sumstats['dH_min']
    Tm_t = true_norm[:, 1] * (sumstats['Tm_max'] - sumstats['Tm_min']) + sumstats['Tm_min']

    dG_p = dH_p * (1.0 - (273.15 + 37.0) / (273.15 + Tm_p))
    dG_t = dH_t * (1.0 - (273.15 + 37.0) / (273.15 + Tm_t))

    metrics = {}
    for tag, p, t in [('dH', dH_p, dH_t), ('Tm', Tm_p, Tm_t), ('dG_37', dG_p, dG_t)]:
        mask = np.isfinite(t) & np.isfinite(p)
        if mask.sum() < 2:
            metrics[f'{tag}_mae'] = metrics[f'{tag}_rmse'] = metrics[f'{tag}_r2'] = float('nan')
        else:
            diff = p[mask] - t[mask]
            metrics[f'{tag}_mae']  = float(np.mean(np.abs(diff)))
            metrics[f'{tag}_rmse'] = float(np.sqrt(np.mean(diff ** 2)))
            metrics[f'{tag}_r2']   = float(r2_score(t[mask], p[mask]))
    return metrics, dH_p, Tm_p, dH_t, Tm_t


@torch.no_grad()
def evaluate(model, loader, sumstats, device):
    model.eval()
    preds, trues = [], []
    for x1, x2, y in loader:
        preds.append(model(x1.to(device), x2.to(device)).cpu())
        trues.append(y)
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    return compute_metrics(preds, trues, sumstats)


print('Metrics helpers defined.')

Metrics helpers defined.


In [8]:
# ── 8. Training Loop ──────────────────────────────────────────────────────────

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'],
                       weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config['n_epoch'], eta_min=1e-5
)

history = {
    'train_loss': [],
    'val_dH_mae': [], 'val_Tm_mae': [], 'val_dG_mae': [],
    'val_dH_rmse': [], 'val_Tm_rmse': [],
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs('out', exist_ok=True)

_run_name = f"Hybrid_gru{config['gru_hidden']}x{config['gru_layers']}_cnn128"
_wandb_kw = dict(project=config['wandb_project'], name=_run_name, config=config, reinit=True)
_mode     = os.environ.get('WANDB_MODE', '').strip().lower()
if _mode in ('offline', 'disabled'):
    run = wandb.init(mode=_mode, **_wandb_kw)
else:
    try:
        run = wandb.init(**_wandb_kw)
    except Exception as _e:
        print(f'WandB online init failed ({_e}), falling back to offline.')
        run = wandb.init(mode='offline', **_wandb_kw)
print(f'WandB run: {run.name}  |  mode: {run.settings.mode}')

best_val_dG = float('inf')
start_time  = time.time()

for epoch in range(config['n_epoch']):
    model.train()
    train_loss = 0.0
    for x1, x2, y in train_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x1, x2), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        train_loss += loss.item() * x1.size(0)
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    val_metrics, _, _, _, _ = evaluate(model, val_loader, sumstats, device)

    history['train_loss'].append(train_loss)
    history['val_dH_mae'].append(val_metrics['dH_mae'])
    history['val_Tm_mae'].append(val_metrics['Tm_mae'])
    history['val_dG_mae'].append(val_metrics['dG_37_mae'])
    history['val_dH_rmse'].append(val_metrics['dH_rmse'])
    history['val_Tm_rmse'].append(val_metrics['Tm_rmse'])

    wandb.log({
        'epoch': epoch, 'train_loss': train_loss,
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'lr': scheduler.get_last_lr()[0],
    })

    if val_metrics['dG_37_mae'] < best_val_dG:
        best_val_dG = val_metrics['dG_37_mae']
        torch.save(model.state_dict(),
                   os.path.join(config['checkpoint_dir'], 'best_hybrid_model.pt'))

    if (epoch + 1) % 20 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Ep {epoch+1:3d}/{config['n_epoch']} "
              f"| loss {train_loss:.4f} "
              f"| dH {val_metrics['dH_mae']:.3f} "
              f"| Tm {val_metrics['Tm_mae']:.3f} "
              f"| dG {val_metrics['dG_37_mae']:.3f} "
              f"| {elapsed:.1f}min")

run.finish()

with open('out/hybrid_history.json', 'w') as f:
    json.dump(history, f)
print(f'\n=== Training complete === Best val dG MAE: {best_val_dG:.4f}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\anant\.netrc.
wandb: Currently logged in as: apati087 (apati087-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run: Hybrid_gru64x2_cnn128  |  mode: online
Ep  20/200 | loss 0.0062 | dH 3.298 | Tm 2.212 | dG 0.204 | 2.8min
Ep  40/200 | loss 0.0043 | dH 3.033 | Tm 1.867 | dG 0.176 | 5.6min
Ep  60/200 | loss 0.0039 | dH 3.375 | Tm 1.930 | dG 0.208 | 8.4min
Ep  80/200 | loss 0.0035 | dH 2.922 | Tm 1.770 | dG 0.170 | 11.2min
Ep 100/200 | loss 0.0033 | dH 3.011 | Tm 1.687 | dG 0.171 | 14.2min
Ep 120/200 | loss 0.0030 | dH 2.879 | Tm 1.629 | dG 0.162 | 17.0min
Ep 140/200 | loss 0.0028 | dH 2.858 | Tm 1.547 | dG 0.161 | 19.9min
Ep 160/200 | loss 0.0026 | dH 2.831 | Tm 1.511 | dG 0.158 | 22.7min
Ep 180/200 | loss 0.0024 | dH 2.838 | Tm 1.522 | dG 0.156 | 25.6min
Ep 200/200 | loss 0.0024 | dH 2.846 | Tm 1.512 | dG 0.156 | 28.5min


epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇██████
lr,█████████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train_loss,█▇▇▆▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_mae,█▄▅▄▃▃▃▃▂▃▂▂▂▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_r2,▁▁▂▂▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇███████████████
val_Tm_rmse,█▅▅▅▅▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁
val_dG_37_mae,█▆▄▄▄▃▂▄▃▂▂▂▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dG_37_r2,▁▂▃▄▆▇████▇▇████████████████████████████
val_dG_37_rmse,█▄▃▃▂▂▂▂▂▁▂▂▁▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dH_mae,▆█▅█▃▃▄▄▂▃▂▂▂▃▁▂▂▁▂▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...



=== Training complete === Best val dG MAE: 0.1547


In [9]:
# ── 9. Final Evaluation ───────────────────────────────────────────────────────

model.load_state_dict(torch.load(
    os.path.join(config['checkpoint_dir'], 'best_hybrid_model.pt'), map_location=device
))

val_metrics,  dH_vp, Tm_vp, dH_vt, Tm_vt = evaluate(model, val_loader,  sumstats, device)
test_metrics, dH_tp, Tm_tp, dH_tt, Tm_tt = evaluate(model, test_loader, sumstats, device)

print('=== Validation Set Results (arr) ===')
for tag, key in [('dH', 'dH'), ('Tm', 'Tm'), ('dG₃₇', 'dG_37')]:
    print(f'  {tag}   MAE {val_metrics[f"{key}_mae"]:.3f}  RMSE {val_metrics[f"{key}_rmse"]:.3f}  R² {val_metrics[f"{key}_r2"]:.3f}')

print('\n=== Test Set Results (arr) ===')
for tag, key in [('dH', 'dH'), ('Tm', 'Tm'), ('dG₃₇', 'dG_37')]:
    print(f'  {tag}   MAE {test_metrics[f"{key}_mae"]:.3f}  RMSE {test_metrics[f"{key}_rmse"]:.3f}  R² {test_metrics[f"{key}_r2"]:.3f}')

eval_df = pd.DataFrame({'dH_pred': dH_vp, 'dH_true': dH_vt,
                         'Tm_pred': Tm_vp, 'Tm_true': Tm_vt})
eval_df['dG_pred'] = eval_df['dH_pred'] * (1 - 310.15 / (273.15 + eval_df['Tm_pred']))
eval_df['dG_true'] = eval_df['dH_true'] * (1 - 310.15 / (273.15 + eval_df['Tm_true']))
eval_df.to_csv('out/hybrid_val_eval.csv', index=False)

run_log = dict(
    experiment_id='E5', model='HybridCNNRNN', config=config,
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
    val_metrics=val_metrics, test_metrics=test_metrics,
    best_checkpoint=os.path.join(config['checkpoint_dir'], 'best_hybrid_model.pt'),
)
with open('out/hybrid_run_log.json', 'w') as f:
    json.dump(run_log, f, indent=2)
print('\nRun log: out/hybrid_run_log.json')

=== Validation Set Results (arr) ===
  dH   MAE 2.814  RMSE 3.884  R² 0.876
  Tm   MAE 1.530  RMSE 2.177  R² 0.959
  dG₃₇   MAE 0.155  RMSE 0.223  R² 0.953

=== Test Set Results (arr) ===
  dH   MAE 2.742  RMSE 3.773  R² 0.883
  Tm   MAE 1.588  RMSE 2.325  R² 0.952
  dG₃₇   MAE 0.153  RMSE 0.220  R² 0.955

Run log: out/hybrid_run_log.json


In [10]:
# ── 10. Convergence Curves (F2 contribution) ──────────────────────────────────

os.makedirs('out/figures', exist_ok=True)
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor='#f8f9fa')
for ax, (key, ylabel, title) in zip(axes, [
    ('val_dH_mae', 'Val dH MAE (kcal/mol)', 'ΔH'),
    ('val_Tm_mae', 'Val Tm MAE (°C)',         'Tm'),
    ('val_dG_mae', 'Val ΔG₃₇ MAE (kcal/mol)', 'ΔG₃₇'),
]):
    ax.plot(epochs, history[key], color=COLORS['Hybrid'], lw=2, label='Hybrid (E5)')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle('Figure F2 (partial) — Hybrid CNN-RNN Validation Convergence',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/hybrid_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/hybrid_convergence.png')

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0, flags=flags)


Saved: out/figures/hybrid_convergence.png


C:\Users\anant\AppData\Local\Temp\ipykernel_38192\675011866.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [11]:
# ── 11. Scatter Plots — Predicted vs Measured (F3 contribution) ───────────────

AXIS_LIMITS = {'dH': (-55, -5), 'Tm': (20, 60), 'dG_37': (-7, 5)}

dG_vp = dH_vp * (1 - 310.15 / (273.15 + Tm_vp))
dG_vt = dH_vt * (1 - 310.15 / (273.15 + Tm_vt))

fig, axes = plt.subplots(1, 3, figsize=(14, 5), facecolor='#f8f9fa')
for ax, (pred_arr, true_arr, tag, unit) in zip(axes, [
    (dH_vp, dH_vt, 'dH',    'kcal/mol'),
    (Tm_vp, Tm_vt, 'Tm',    '°C'),
    (dG_vp, dG_vt, 'dG_37', 'kcal/mol'),
]):
    lim = AXIS_LIMITS[tag]
    ax.scatter(true_arr, pred_arr, s=4, alpha=0.4, color=COLORS['Hybrid'], rasterized=True)
    ax.plot(lim, lim, 'k--', alpha=0.3, lw=1.5)
    mask = np.isfinite(pred_arr) & np.isfinite(true_arr)
    mae  = np.mean(np.abs(pred_arr[mask] - true_arr[mask]))
    rmse = np.sqrt(np.mean((pred_arr[mask] - true_arr[mask]) ** 2))
    r2   = r2_score(true_arr[mask], pred_arr[mask])
    ax.text(0.05, 0.93, f'MAE={mae:.3f}\nRMSE={rmse:.3f}\nR²={r2:.3f}',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f'Measured {tag} ({unit})')
    ax.set_ylabel(f'Predicted {tag} ({unit})')
    ax.set_title(f'Hybrid — {tag}', fontsize=11, fontweight='bold')
    sns.despine(ax=ax)

fig.suptitle('Figure F3 (partial) — Hybrid CNN-RNN: Predicted vs Measured (Validation)',
             fontsize=12)
plt.tight_layout()
plt.savefig('out/figures/hybrid_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/hybrid_scatter.png')

Saved: out/figures/hybrid_scatter.png


C:\Users\anant\AppData\Local\Temp\ipykernel_38192\3459613113.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [12]:
# ── 12. Branch Ablation — Which modality matters more? ────────────────────────
#
# Key thesis analysis: run each branch independently by zeroing out the other.
# This shows whether the Hybrid gain comes from the RNN, CNN, or their interaction.

@torch.no_grad()
def evaluate_ablation(model, loader, sumstats, device, zero_rnn=False, zero_cnn=False):
    """Run hybrid model with one branch zeroed to measure individual contributions."""
    model.eval()
    preds, trues = [], []
    for x1, x2, y in loader:
        x1, x2 = x1.to(device), x2.to(device)
        if zero_rnn:
            x1 = torch.zeros_like(x1)
        if zero_cnn:
            x2 = torch.zeros_like(x2)
        preds.append(model(x1, x2).cpu())
        trues.append(y)
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    metrics, *_ = compute_metrics(preds, trues, sumstats)
    return metrics


full_m    = evaluate_ablation(model, val_loader, sumstats, device)
cnn_only  = evaluate_ablation(model, val_loader, sumstats, device, zero_rnn=True)
rnn_only  = evaluate_ablation(model, val_loader, sumstats, device, zero_cnn=True)

print('=== Branch Ablation (Val dG₃₇ MAE) ===')
print(f'  Full Hybrid:  {full_m["dG_37_mae"]:.3f} kcal/mol')
print(f'  CNN only:     {cnn_only["dG_37_mae"]:.3f} kcal/mol  (RNN zeroed)')
print(f'  RNN only:     {rnn_only["dG_37_mae"]:.3f} kcal/mol  (CNN zeroed)')

# Bar chart — thesis Figure F7 contribution
labels  = ['Full Hybrid\n(CNN + RNN)', 'CNN branch\nonly', 'RNN branch\nonly']
dG_vals = [full_m['dG_37_mae'], cnn_only['dG_37_mae'], rnn_only['dG_37_mae']]
colors  = [COLORS['Hybrid'], COLORS['2D_CNN'], COLORS['1D_CNN']]

fig, ax = plt.subplots(figsize=(7, 5), facecolor='#f8f9fa')
bars = ax.bar(labels, dG_vals, color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, dG_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Val ΔG₃₇ MAE (kcal/mol)', fontsize=11)
ax.set_title('Branch Ablation — Hybrid CNN-RNN\n(lower = better)',
             fontsize=11, fontweight='bold')
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('out/figures/hybrid_ablation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/hybrid_ablation.png')

=== Branch Ablation (Val dG₃₇ MAE) ===
  Full Hybrid:  0.155 kcal/mol
  CNN only:     0.390 kcal/mol  (RNN zeroed)
  RNN only:     1.077 kcal/mol  (CNN zeroed)


c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0, flags=flags)


Saved: out/figures/hybrid_ablation.png


C:\Users\anant\AppData\Local\Temp\ipykernel_38192\2242420495.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
